In [ ]:
import MDAnalysis as mda
import numpy as np
from MDAnalysis.lib.distances import capped_distance
import nglview as nv

In [ ]:
u_atoms = {}

proteins = ["hpl2", "met2", "lin13", "lin65"]
z_coordinates = [0, 0, -500, 500]


for prot, z_coordinate in zip(proteins, z_coordinates):
    try:
        print(f"Working on protein {prot}...")
        u = mda.Universe(f"input_structures/single_comp_structures/{prot}_config_260.15K.pdb")
        
        prot_atoms = u.atoms
        
        # assign individual segment IDs to all chains
        
        for segment in prot_atoms.segments:


        # Set box dimensions
        u_atoms[prot].atoms.dimensions = [600, 600, 3000, 90, 90, 90]
        center = u_atoms[prot].center_of_geometry()
        
        # Center the structures in the middle of the box and displace them on the z axis
        u_atoms[prot].translate([-center[0] + 300, -center[1] + 300, z_coordinate])
        
        # Create segment name (first 4 characters of protein name, with numbering if needed)
        seg_name = prot.upper()[:4]
        if len(seg_name) < 4:
            seg_name = seg_name.ljust(4, 'X')
        u_atoms[prot].atoms.segments.segids = seg_name
        
    except FileNotFoundError:
        print(f"Error: Could not find file for protein {prot}")
        continue

# Merge the structures of all proteins
merged = mda.Merge(*[u_atoms[prot] for prot in proteins if prot in u_atoms])
merged.atoms.dimensions = [600, 600, 3000, 90, 90, 90]
print("AtomGroups merged.")

# Process each segment
unique_segs = np.unique(merged.atoms.segids)
all_bonds = []

for seg in unique_segs:
    print(f"Processing chains for segment: {seg}")
    seg_atoms = merged.select_atoms(f"segid {seg}")
    
    # Get individual chains and store them
    chains = get_individual_chains(seg_atoms)
    
    # Create bonds for each chain
    for chain_id, chain_atoms in chains.items():
        seg_bonds = intelligent_guess_bonds(chain_atoms)
        all_bonds.extend(seg_bonds)

print(f"Adding {len(all_bonds)} bonds to the system...")
merged.add_TopologyAttr("bonds", all_bonds)

# Create output directory if it doesn't exist
import os
os.makedirs("output", exist_ok=True)

# Create a pdb file
merged.atoms.write(f"output/all_comps_merged_260.15.pdb")
print("Output file created successfully.")